# Tarkov Aim Lab - Exploration Notebook

Use this notebook to:
1. Test individual components
2. Visualize detection results
3. Tune thresholds
4. Explore footage frame-by-frame

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

# For inline video display
from IPython.display import display, Image, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

from src.detection import CrosshairTracker, FlickDetector, EnemyDetector
from src.detection import visualize_tracking, visualize_detections
from src.analysis import FlickAnalyzer, calculate_sensitivity_adjustment

In [ ]:
# Load config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)
    
print("Player settings:")
print(f"  Sensitivity: {config['player']['sensitivity']}")
print(f"  ADS Multiplier: {config['player']['ads_multiplier']}")
print(f"  FOV (hip/ads): {config['player']['fov']['hip']}/{config['player']['fov']['ads']}")

## 1. Load a Test Video

Point this to an Outplayed clip from an Arena match.

In [ ]:
# UPDATE THIS PATH to point to your Outplayed recordings
VIDEO_PATH = r"D:\Outplayed\EscapeFromTarkov\your_clip_here.mp4"

# Or list available clips
outplayed_dir = Path(r"D:\Outplayed\EscapeFromTarkov")
if outplayed_dir.exists():
    clips = list(outplayed_dir.glob("*.mp4"))
    print(f"Found {len(clips)} clips:")
    for c in clips[:10]:  # Show first 10
        print(f"  {c.name}")
else:
    print(f"Directory not found: {outplayed_dir}")
    print("Update the path to your Outplayed folder")

In [ ]:
# Load video
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"ERROR: Could not open {VIDEO_PATH}")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    
    print(f"Video loaded: {width}x{height} @ {fps:.1f}fps")
    print(f"Duration: {duration:.1f}s ({total_frames} frames)")

## 2. Test Crosshair Motion Detection

Process a segment and visualize the velocity over time.

In [ ]:
# Initialize tracker
tracker = CrosshairTracker(
    frame_width=width,
    frame_height=height,
    roi_width_pct=0.1,
    roi_height_pct=0.1,
    velocity_threshold=15.0  # Tune this!
)

# Process first N frames
N_FRAMES = 500  # ~8 seconds at 60fps

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # Reset to start

velocities = []
for i in range(N_FRAMES):
    ret, frame = cap.read()
    if not ret:
        break
    state = tracker.process_frame(frame)
    velocities.append(state.velocity)

velocities = np.array(velocities)
print(f"Processed {len(velocities)} frames")

In [ ]:
# Plot velocity over time
fig, ax = plt.subplots(figsize=(14, 5))

time = np.arange(len(velocities)) / fps  # Convert to seconds
ax.plot(time, velocities, 'b-', linewidth=0.5)
ax.axhline(y=15, color='r', linestyle='--', label='Threshold')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Velocity (px/frame)')
ax.set_title('Crosshair Motion Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Stats
print(f"\nVelocity stats:")
print(f"  Mean: {np.mean(velocities):.2f}")
print(f"  Max: {np.max(velocities):.2f}")
print(f"  Frames above threshold: {np.sum(velocities > 15)} ({100*np.mean(velocities > 15):.1f}%)")

## 3. Test Flick Detection

In [ ]:
# Run flick detector on velocity data
flick_detector = FlickDetector()
flicks = flick_detector.process_velocity_series(velocities)

print(f"Detected {len(flicks)} flicks:")
for f in flicks[:10]:  # Show first 10
    start_time = f.reaction_frame / fps
    end_time = f.termination_frame / fps
    print(f"  {start_time:.2f}s - {end_time:.2f}s (dur={f.duration_frames}f, peak={f.peak_velocity:.1f})")

In [ ]:
# Visualize flicks on the velocity plot
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(time, velocities, 'b-', linewidth=0.5, alpha=0.7)
ax.axhline(y=15, color='r', linestyle='--', alpha=0.5)

# Highlight flick regions
for f in flicks:
    start = f.reaction_frame / fps
    end = f.termination_frame / fps
    ax.axvspan(start, end, color='green', alpha=0.3)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Velocity (px/frame)')
ax.set_title(f'Detected Flicks (n={len(flicks)}) - Green regions')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Test Enemy Detection (YOLO)

In [ ]:
# Initialize YOLO detector
enemy_detector = EnemyDetector(
    model_path='yolov8n.pt',  # Will download if not present
    confidence_threshold=0.5,
    device='cuda'  # or 'cpu'
)

In [ ]:
# Jump to a specific frame and detect
FRAME_NUM = 300  # Change this to explore different frames

cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_NUM)
ret, frame = cap.read()

if ret:
    detections = enemy_detector.detect(frame)
    screen_center = (width // 2, height // 2)
    
    print(f"Frame {FRAME_NUM}: Found {len(detections)} persons")
    for d in detections:
        print(f"  Center: {d.center}, Conf: {d.confidence:.2f}")
    
    # Visualize
    vis = visualize_detections(frame, detections, screen_center)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(vis_rgb)
    plt.title(f'Frame {FRAME_NUM} - {len(detections)} detections')
    plt.axis('off')
    plt.show()

## 5. Frame-by-Frame Explorer

Interactive widget to step through frames.

In [ ]:
def show_frame(frame_num):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    
    if ret:
        # Detect enemies
        detections = enemy_detector.detect(frame)
        screen_center = (width // 2, height // 2)
        
        # Visualize
        vis = visualize_detections(frame, detections, screen_center)
        vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
        
        # Get velocity if available
        vel_text = ""
        if frame_num < len(velocities):
            vel_text = f" | Velocity: {velocities[frame_num]:.1f}"
        
        plt.figure(figsize=(12, 8))
        plt.imshow(vis_rgb)
        plt.title(f'Frame {frame_num}{vel_text}')
        plt.axis('off')
        plt.show()

# Create slider
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=total_frames-1,
    step=1,
    description='Frame:',
    continuous_update=False
)

widgets.interact(show_frame, frame_num=slider)

## 6. Tune Detection Thresholds

Adjust these values based on what you see above.

In [ ]:
# Threshold tuning
# Based on the velocity plot above, adjust these:

VELOCITY_THRESHOLD = 15.0  # Pixels/frame to count as "moving"
SETTLE_FRAMES = 3          # Frames of stillness to end a flick
MIN_FLICK_FRAMES = 2       # Minimum flick duration
MAX_FLICK_FRAMES = 30      # Maximum flick duration

# Re-run detection with new thresholds
from src.detection.events import FlickDetectorConfig

new_config = FlickDetectorConfig(
    velocity_threshold=VELOCITY_THRESHOLD,
    settle_frames=SETTLE_FRAMES,
    min_flick_frames=MIN_FLICK_FRAMES,
    max_flick_frames=MAX_FLICK_FRAMES
)

tuned_detector = FlickDetector(config=new_config)
tuned_flicks = tuned_detector.process_velocity_series(velocities)

print(f"With tuned thresholds: {len(tuned_flicks)} flicks (was {len(flicks)})")

## 7. Save Good Thresholds to Config

In [ ]:
# Once you've found good values, update config.yaml:
print("Update these in config.yaml:")
print(f"")
print(f"analysis:")
print(f"  flick_velocity_threshold: {VELOCITY_THRESHOLD}")
print(f"  settle_frames: {SETTLE_FRAMES}")
print(f"  min_flick_frames: {MIN_FLICK_FRAMES}")

In [ ]:
# Cleanup
cap.release()